In [0]:
# Databricks notebook source

from pyspark.sql import functions as F

In [0]:
RAW_PATH = "/Volumes/energy/raw/fingrid/consumption/"

Read in raw data

In [0]:
raw_data_df = (
    spark.read.option("multiline", "true")
    .json(RAW_PATH)
)
display(raw_data_df)

Create bronze layer data frame

In [0]:
bronze_df = (
    raw_data_df
    .withColumn("datasetId", F.col("datasetId").cast("integer"))
    .withColumn("startTime", F.to_timestamp("startTime"))
    .withColumn("endTime", F.to_timestamp("endTime"))
)

In [0]:
display(bronze_df)

Use right catalog and schema

In [0]:
%sql

USE CATALOG energy;
USE SCHEMA RAW;

Create schemas for bronze, silver and gold layer

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS energy.bronze;
CREATE SCHEMA IF NOT EXISTS energy.silver;
CREATE SCHEMA IF NOT EXISTS energy.gold;

Save bronze consumption data frame to bronze consumption table in schema bronze

In [0]:
bronze_df.write.format("delta").mode("append").saveAsTable("energy.bronze.consumption")

Check that the table energy.bronze.consumption is saved correctly

In [0]:
%sql

SELECT * FROM energy.bronze.consumption LIMIT 20